In [1]:
"""
exp13 Step 04: Execution And Episode Robustness Of E1M1 On DEV, And The Verdict (PROTOCOL.md §4 step 04, §8)

DEV = 2015-04-17 -> 2026-04-15 (SPY and VIX cut at 2026-04-15). Reproduction asserts first.
    E1  delayed execution: every trade (exits and re-entries) fills at the fill bar of the NEXT session.
    E2  costs: slippage 0, 1, 2 and 5 cents per share per side (buy-and-hold recomputed with the same slippage; the
        verdict compares with buy-and-hold under exp12's costs, the stricter comparison).
    E3  episodes: product of S/R without the best and the two best episodes (by log(S/R)); log excess per calendar year;
        log excess on 2015-04-17 -> 2020-12-31 and 2021-01-01 -> 2026-04-15; the share of the total log excess from the
        episodes overlapping 2020-02-01 -> 2020-06-30; the split of the product of S/R by re-entry reason.
Then the pre-registered verdict (PROTOCOL.md §8), applied once, from the saved outputs of steps 01-03 and this step:
"ROBUST CANDIDATE PENDING REVIEW" only if R1-R5 all hold, otherwise "E1M1 IS NOT ROBUST". No success claim either way.
RUN_MODE: "validation_only" only. Outputs: store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/.
Trials: 2 (stage "robustness": delayed execution, 5-cent costs) + 1 summary.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp13_e1m1_robustness import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION AND REPLAY FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
from so.core.continuous_replay import get_episode_scorecard_pdf
# IMPORT exp12's REPORTS (THE SAME DEFINITIONS AS exp12's CONCENTRATION REPORTS)
from experiments.exp12_vix_fear_reentry.rules import get_vix_scorecard_pdf, get_concentration_dict, get_yearly_log_excess_pdf
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp13_e1m1_robustness.robustness import get_span_data_dict, get_dev_period_pdf, check_last_date_dict, get_dev_reproduction_dict, get_delayed_rule_tuple, \
    replay_rule_dict, get_path_metric_dict, get_span_log_excess_pdf, get_reason_split_pdf, get_verdict_dict, check_step_trial_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Run Mode:\t{RUN_MODE_STR}\tExperiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP AND THE FOLDERS OF THE EARLIER STEPS (READ FOR THE VERDICT)
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step04_execution_episode_data")
step_path_dict = {step_str: paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, folder_str) for step_str, folder_str in
                  [("step01", "step01_stress_data"), ("step02", "step02_neighbourhood_data"), ("step03", "step03_placebo_data")]}
print(f"Output folder:\t{output_path_str}")
# WARN ABOUT EARLIER ENTRIES OF THIS STEP (THE "robustness" STAGE ALSO HOLDS STEP 02's ENTRIES; THE NOTE PREFIX SEPARATES THEM)
trial_log_pdf = read_trial_log_pdf()
previous_step04_count = int((trial_log_pdf["experiment"].eq(exp_config.EXPERIMENT_NAME) & trial_log_pdf["note"].fillna("").str.startswith("exp13 step04")).sum()) if len(trial_log_pdf) else 0
print(f"⚠️ The trial log already holds {previous_step04_count} exp13 step04 entries: a rerun ADDS trials" if previous_step04_count else "ℹ️ No earlier exp13 step04 entry in the trial log.")

Run Mode:	validation_only	Experiment:	exp13_e1m1_robustness	Protocol:	1.0	Config Hash:	c2e293d595
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/
ℹ️ No earlier exp13 step04 entry in the trial log.


In [4]:
# CALL FUNCTION TO READ exp12's 44 REPLAY PERIODS (DATES ONLY) AND LOAD SPY AND THE VIX UP TO THEIR END (2026-04-15)
dev_period_pdf = get_dev_period_pdf(CHECK_REAL_DATA_BOOL)
dev_cutoff_date_str = str(dev_period_pdf["period_end"].iloc[-1])
dev_data_dict = get_span_data_dict(dev_cutoff_date_str)
ohlcv_array_dict, daily_pdf, vix_feature_pdf = dev_data_dict["ohlcv_array_dict"], dev_data_dict["daily_pdf"], dev_data_dict["vix_feature_pdf"]
print(f"DEV:\t{dev_period_pdf['period_start'].iloc[0]} -> {dev_cutoff_date_str}\t({len(dev_period_pdf)} periods)\tSPY sessions loaded {len(daily_pdf):,} from {dev_data_dict['first_spy_date']}")
dev_last_date_dict = check_last_date_dict(dev_data_dict["last_date_dict"], dev_cutoff_date_str, CHECK_REAL_DATA_BOOL)

DEV:	2015-04-17 -> 2026-04-15	(44 periods)	SPY sessions loaded 5,353 from 2005-01-03
Last date loaded:	spy	2026-04-15	(cutoff 2026-04-15)
Last date loaded:	vix_daily	2026-04-15	(cutoff 2026-04-15)
Last date loaded:	vix3m_daily	2026-04-15	(cutoff 2026-04-15)


In [5]:
"""
Reproduction (before any exp13 number).
"""
# CALL FUNCTION TO REPLAY AND ASSERT THE REPRODUCTION PATHS
dev_reproduction_dict = get_dev_reproduction_dict(dev_data_dict, dev_period_pdf, CHECK_REAL_DATA_BOOL)
print(f"Reproduction:\t{dev_reproduction_dict['reproduction_dict']}")
print("Reproduction asserts:\tPASSED") if CHECK_REAL_DATA_BOOL else None
date1, date2 = dev_reproduction_dict["date1"], dev_reproduction_dict["date2"]
frozen_simulation_dict, buy_hold_dict = dev_reproduction_dict["frozen_simulation_dict"], dev_reproduction_dict["buy_hold_dict"]
frozen_metric_dict = get_path_metric_dict(frozen_simulation_dict, buy_hold_dict, dev_period_pdf, ohlcv_array_dict)

Reproduction:	{'dev_start': '2015-04-17', 'dev_end': '2026-04-15', 'e1m1_total_pct': 314.8, 'e1m1_exit_count': 35, 'e1_total_pct': 108.03, 'e1_exit_count': 35, 'buy_hold_pct': 235.39}
Reproduction asserts:	PASSED


In [6]:
"""
E1 delayed execution and E2 costs. Each path against buy-and-hold under exp12's costs (excess_total_return) and, for the
cost levels, against buy-and-hold paying the same slippage (excess_vs_same_cost_buy_hold).
"""
# LIST TO HOLD THE ROWS
execution_dict_list = [{"variant": "frozen", "slippage_per_share": config.ENTRY_SLIPPAGE_PER_SHARE, **frozen_metric_dict,
                        "same_cost_buy_hold_total_return": buy_hold_dict["metric_dict"]["total_return"]}]
# E1: ONE-SESSION-LATE EXECUTION
delayed_simulation_dict = replay_rule_dict(ohlcv_array_dict, get_delayed_rule_tuple(dev_reproduction_dict["frozen_rule_tuple"], exp_config.DELAY_SESSION_COUNT), date1, date2)
execution_dict_list.append({"variant": "delayed_1_session", "slippage_per_share": config.ENTRY_SLIPPAGE_PER_SHARE,
                            **get_path_metric_dict(delayed_simulation_dict, buy_hold_dict, dev_period_pdf, ohlcv_array_dict),
                            "same_cost_buy_hold_total_return": buy_hold_dict["metric_dict"]["total_return"]})
# E2: THE SLIPPAGE LEVELS
for slippage in exp_config.COST_SLIPPAGE_LIST:
    cost_simulation_dict = replay_rule_dict(ohlcv_array_dict, dev_reproduction_dict["frozen_rule_tuple"], date1, date2, entry_slippage_in=slippage, market_exit_slippage_in=slippage)
    cost_buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, date1, date2, entry_slippage_in=slippage, market_exit_slippage_in=slippage)
    execution_dict_list.append({"variant": f"slippage_{int(round(100 * slippage))}c", "slippage_per_share": slippage,
                                **get_path_metric_dict(cost_simulation_dict, buy_hold_dict, dev_period_pdf, ohlcv_array_dict),
                                "same_cost_buy_hold_total_return": cost_buy_hold_dict["metric_dict"]["total_return"]})
execution_pdf = pd.DataFrame(execution_dict_list)
execution_pdf["excess_vs_same_cost_buy_hold"] = execution_pdf["total_return"] - execution_pdf["same_cost_buy_hold_total_return"]
# ASSERT THAT THE 1-CENT LEVEL IS THE FROZEN PATH
assert execution_pdf.loc[execution_pdf["variant"] == "slippage_1c", "total_return"].iloc[0] == frozen_metric_dict["total_return"]
# PREVIEW DATAFRAME
execution_pdf[["variant", "slippage_per_share", "total_return", "buy_hold_total_return", "excess_total_return", "same_cost_buy_hold_total_return", "excess_vs_same_cost_buy_hold",
               "annualized_log_excess", "log_excess_ci_low_6m", "log_excess_ci_high_6m", "log_excess_ci_low_1m", "log_excess_ci_high_1m", "in_market_pct", "exit_count",
               "sharpe_ratio", "max_drawdown", "product_s_over_r"]].round(4)

,variant,slippage_per_share,total_return,buy_hold_total_return,excess_total_return,same_cost_buy_hold_total_return,excess_vs_same_cost_buy_hold,annualized_log_excess,log_excess_ci_low_6m,log_excess_ci_high_6m,log_excess_ci_low_1m,log_excess_ci_high_1m,in_market_pct,exit_count,sharpe_ratio,max_drawdown,product_s_over_r
0,frozen,0.01,3.1480,2.3539,0.7941,2.3539,0.7941,0.0192,-0.0113,0.0603,-0.0175,0.0637,0.9403,35,0.9273,-0.2472,1.2383
1,delayed_1_session,0.01,3.0615,2.3539,0.7076,2.3539,0.7076,0.0173,-0.0171,0.0645,-0.0234,0.0703,0.9403,35,0.9177,-0.2361,1.2128
2,slippage_0c,0.00,3.1588,2.3539,0.8049,2.3540,0.8048,0.0194,-0.0110,0.0607,-0.0173,0.0640,0.9403,35,0.9289,-0.2469,1.2412
3,slippage_1c,0.01,3.1480,2.3539,0.7941,2.3539,0.7941,0.0192,-0.0113,0.0603,-0.0175,0.0637,0.9403,35,0.9273,-0.2472,1.2383
4,slippage_2c,0.02,3.1376,2.3539,0.7837,2.3538,0.7838,0.0189,-0.0115,0.0599,-0.0177,0.0635,0.9403,35,0.9257,-0.2474,1.2355
5,slippage_5c,0.05,3.1085,2.3539,0.7546,2.3535,0.7550,0.0183,-0.0122,0.0589,-0.0182,0.0630,0.9403,35,0.9215,-0.2479,1.2269


In [7]:
"""
E3 episodes of the frozen E1M1 on DEV: the scorecard (re-entry reason, VIX at the exit and at the buy-back), the product
of S/R without the best one and two episodes, the halves, the 2020 share and the split by re-entry reason.
"""
# BUILD THE SCORECARD AND THE REPORTS
scorecard_pdf = get_vix_scorecard_pdf(get_episode_scorecard_pdf(frozen_simulation_dict, ohlcv_array_dict), frozen_simulation_dict, vix_feature_pdf)
path_equity_pdf, buy_hold_equity_pdf = frozen_simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"]
concentration_dict = get_concentration_dict(scorecard_pdf, path_equity_pdf, buy_hold_equity_pdf, exp_config.CONCENTRATION_WINDOW_START_STR, exp_config.CONCENTRATION_WINDOW_END_STR)
year_pdf = get_yearly_log_excess_pdf(path_equity_pdf, buy_hold_equity_pdf)
half_pdf = get_span_log_excess_pdf(path_equity_pdf, buy_hold_equity_pdf, exp_config.HALF_SPAN_LIST)
reason_pdf = get_reason_split_pdf(scorecard_pdf)
top_pdf = scorecard_pdf.sort_values("log_share_gain", ascending=False).head(2)
# DISPLAY THE REPORTS
print(f"Final equity / buy & hold:\t{concentration_dict['equity_ratio']:.4f}\tproduct of S/R:\t{concentration_dict['product_s_over_r']:.4f}\t({concentration_dict['episode_count']} episodes)")
print(f"    without the best episode:\tequity ratio {concentration_dict['equity_ratio_without_top1']:.4f}\tproduct {concentration_dict['product_s_over_r_without_top1']:.4f}\tbeats buy & hold: {concentration_dict['beats_buy_hold_without_top1']}")
print(f"    without the two best:\tequity ratio {concentration_dict['equity_ratio_without_top2']:.4f}\tproduct {concentration_dict['product_s_over_r_without_top2']:.4f}\tbeats buy & hold: {concentration_dict['beats_buy_hold_without_top2']}")
print(f"    the two best episodes:\t{[(str(row['exit_date']), str(row['reentry_date']), round(row['s_over_r'], 4), row['reentry_trigger']) for _, row in top_pdf.iterrows()]}")
print(f"Episodes overlapping {exp_config.CONCENTRATION_WINDOW_START_STR} -> {exp_config.CONCENTRATION_WINDOW_END_STR}:\t{concentration_dict['window_episode_count']}\t"
      f"sum of log(S/R) {concentration_dict['window_log_share_gain']:.4f}\ttotal log excess {concentration_dict['total_log_excess']:.4f}\tshare {concentration_dict['window_share_of_log_excess']:.2f}")
print("Halves:")
display(half_pdf.round(4))
print("Split of the product of S/R by re-entry reason:")
display(reason_pdf.round(4))
print("Log excess per calendar year:")
display(year_pdf.round(4))
# DISPLAY THE SCORECARD
scorecard_pdf[["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "log_share_gain", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_at_reentry",
               "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]].round(4)

Final equity / buy & hold:	1.2368	product of S/R:	1.2383	(35 episodes)
    without the best episode:	equity ratio 0.9775	product 0.9788	beats buy & hold: False
    without the two best:	equity ratio 0.8945	product 0.8956	beats buy & hold: False
    the two best episodes:	[('2020-03-05', '2020-03-16', 1.2652, 'M1'), ('2018-12-04', '2018-12-27', 1.0928, 'M1')]
Episodes overlapping 2020-02-01 -> 2020-06-30:	4	sum of log(S/R) 0.1470	total log excess 0.2125	share 0.69
Halves:


,span_start,span_end,session_count,strategy_log_return,buy_hold_log_return,log_excess,annualized_log_excess
0,2015-04-17,2020-12-31,1439,0.7854,0.5836,0.2018,0.0353
1,2021-01-01,2026-04-15,1326,0.6372,0.6265,0.0107,0.0020


Split of the product of S/R by re-entry reason:


,reentry_trigger,episode_count,product_s_over_r,log_share_gain_sum,bought_back_lower_count
0,M1,13,1.8188,0.5982,10
1,original rule,22,0.6808,-0.3844,0


Log excess per calendar year:


,year,session_count,strategy_log_return,buy_hold_log_return,log_excess,in_market_pct
0,2015,180,-0.0581,-0.0219,-0.0363,0.8722
1,2016,252,0.1495,0.0919,0.0577,0.9683
2,2017,251,0.1771,0.1769,0.0002,1.0000
3,2018,251,0.0015,-0.0650,0.0665,0.8486
4,2019,252,0.2190,0.2522,-0.0332,0.9881
5,2020,253,0.2964,0.1495,0.1469,0.9565
6,2021,252,0.2389,0.2390,-0.0000,1.0000
7,2022,251,-0.1823,-0.2164,0.0341,0.8367
8,2023,250,0.1366,0.2173,-0.0807,0.9320
9,2024,252,0.2090,0.2092,-0.0001,1.0000


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,log_share_gain,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,open_at_end
0,2015-07-08,204.61,2015-07-10,207.69,0.9852,-0.0149,2,original rule,16.09,19.97,0.0150,-0.0009,False
1,2015-08-20,204.14,2015-08-27,199.03,1.0257,0.0254,5,M1,15.25,30.32,-0.0251,-0.1065,False
2,2015-11-12,204.86,2015-11-18,208.86,0.9808,-0.0193,4,original rule,16.06,18.84,0.0194,-0.0131,False
3,2015-12-03,205.55,2015-12-04,209.62,0.9806,-0.0196,1,original rule,15.91,18.11,0.0197,-0.0006,False
4,2015-12-09,205.37,2015-12-16,208.07,0.9870,-0.0131,5,original rule,17.60,20.95,0.0130,-0.0264,False
5,2015-12-17,205.16,2015-12-23,206.06,0.9956,-0.0044,4,M1,17.86,16.60,0.0043,-0.0254,False
6,2015-12-30,205.95,2016-01-13,189.09,1.0892,0.0854,9,M1,16.08,22.47,-0.0820,-0.0854,False
7,2016-06-27,199.56,2016-06-28,203.12,0.9825,-0.0177,1,original rule,25.76,23.85,0.0177,-0.0001,False
8,2018-04-02,257.51,2018-04-03,260.63,0.9880,-0.0120,1,original rule,19.97,23.62,0.0120,-0.0026,False
9,2018-10-11,272.50,2018-10-12,276.64,0.9850,-0.0151,1,original rule,22.96,24.98,0.0151,-0.0011,False


In [8]:
"""
The pre-registered verdict (PROTOCOL.md §8), from the saved outputs of steps 01-03 and the numbers above.
"""
# READ THE SAVED OUTPUTS OF STEPS 01-03
stress_pdf = pd.read_csv(f"{step_path_dict['step01']}stress_path_data.csv")
neighbourhood_dict = pd.read_csv(f"{step_path_dict['step02']}neighbourhood_summary_data.csv").iloc[0].to_dict()
placebo_pdf = pd.read_csv(f"{step_path_dict['step03']}placebo_summary_data.csv").set_index("test")
stress_e1m1_row = stress_pdf[(stress_pdf["window"] == "stress") & (stress_pdf["candidate_name"] == "E1M1")].iloc[0]
# COLLECT THE NUMBERS OF THE FIVE CRITERIA
delayed_total = float(execution_pdf.loc[execution_pdf["variant"] == "delayed_1_session", "total_return"].iloc[0])
cost_total = float(execution_pdf.loc[execution_pdf["variant"] == f"slippage_{int(round(100 * exp_config.LOGGED_COST_SLIPPAGE))}c", "total_return"].iloc[0])
dev_buy_hold_total = float(buy_hold_dict["metric_dict"]["total_return"])
verdict_input_dict = {"stress_total": float(stress_e1m1_row["total_return"]), "stress_buy_hold_total": float(stress_e1m1_row["buy_hold_total_return"]),
                      "dev_beat_count": int(neighbourhood_dict["dev_beat_count"]), "stress_beat_count": int(neighbourhood_dict["stress_beat_count"]),
                      "p2": float(placebo_pdf.loc["P2_vix_shift_selection_aware", "p_value"]), "equity_ratio_without_top1": float(concentration_dict["equity_ratio_without_top1"]),
                      "delayed_total": delayed_total, "cost_total": cost_total, "dev_buy_hold_total": dev_buy_hold_total}
# APPLY THE RULE ONCE
verdict_dict = get_verdict_dict(verdict_input_dict["stress_total"], verdict_input_dict["stress_buy_hold_total"], verdict_input_dict["dev_beat_count"], verdict_input_dict["stress_beat_count"],
                                verdict_input_dict["p2"], verdict_input_dict["equity_ratio_without_top1"], delayed_total, cost_total, dev_buy_hold_total)
print(f"R1 STRESS:\tE1M1 {verdict_input_dict['stress_total']:.2%} vs buy & hold {verdict_input_dict['stress_buy_hold_total']:.2%}\t-> {verdict_dict['r1_stress']}")
print(f"R2 PLATEAU:\t{verdict_input_dict['dev_beat_count']} of 36 beat buy & hold on DEV (need {exp_config.R2_DEV_MIN_COUNT}), {verdict_input_dict['stress_beat_count']} on STRESS (need {exp_config.R2_STRESS_MIN_COUNT})\t-> {verdict_dict['r2_plateau']}")
print(f"R3 INFORMATION:\tp2 = {verdict_input_dict['p2']:.4f} (need <= {exp_config.R3_MAX_P})\t-> {verdict_dict['r3_information']}")
print(f"R4 EPISODES:\tequity ratio without the best episode {verdict_input_dict['equity_ratio_without_top1']:.4f} (need > 1)\t-> {verdict_dict['r4_episodes']}")
print(f"R5 EXECUTION:\tdelayed {delayed_total:.2%}, 5-cent {cost_total:.2%} vs buy & hold {dev_buy_hold_total:.2%}\t-> {verdict_dict['r5_execution']}")
print(f"\nVERDICT:\t{verdict_dict['verdict_str']}\tfailed: {verdict_dict['failed_list']}")

R1 STRESS:	E1M1 53.32% vs buy & hold 73.09%	-> False
R2 PLATEAU:	11 of 36 beat buy & hold on DEV (need 24), 3 on STRESS (need 18)	-> False
R3 INFORMATION:	p2 = 0.0880 (need <= 0.05)	-> False
R4 EPISODES:	equity ratio without the best episode 0.9775 (need > 1)	-> False
R5 EXECUTION:	delayed 306.15%, 5-cent 310.85% vs buy & hold 235.39%	-> True

VERDICT:	E1M1 IS NOT ROBUST	failed: ['r1_stress', 'r2_plateau', 'r3_information', 'r4_episodes']


In [9]:
# SAVE THE RESULTS
write_csv_file_to_path(execution_pdf, f"{output_path_str}execution_path_data.csv", "W")
write_csv_file_to_path(scorecard_pdf, f"{output_path_str}episode_scorecard_data.csv", "W")
write_csv_file_to_path(pd.DataFrame([concentration_dict]), f"{output_path_str}concentration_data.csv", "W")
write_csv_file_to_path(year_pdf, f"{output_path_str}yearly_log_excess_data.csv", "W")
write_csv_file_to_path(half_pdf, f"{output_path_str}half_log_excess_data.csv", "W")
write_csv_file_to_path(reason_pdf, f"{output_path_str}reason_split_data.csv", "W")
write_csv_file_to_path(pd.DataFrame([{**verdict_input_dict, **{key: value for key, value in verdict_dict.items() if key != "failed_list"}, "failed_list": ";".join(verdict_dict["failed_list"])}]),
                       f"{output_path_str}verdict_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/execution_path_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/execution_path_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/episode_scorecard_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/episode_scorecard_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step04_execution_episode_data/conce

In [10]:
# CHECK THE TRIAL BUDGET AND THE EXPECTED COUNT BEFORE LOGGING (2 ROBUSTNESS + 1 SUMMARY)
trial_log_pdf = read_trial_log_pdf()
check_step_trial_dict(trial_log_pdf, "step04", 3)
# LOG THE DELAYED AND THE 5-CENT PATHS AS ROBUSTNESS TRIALS (WINDOW COLUMNS = DEV)
logged_dict_list = []
for variant_str in ["delayed_1_session", f"slippage_{int(round(100 * exp_config.LOGGED_COST_SLIPPAGE))}c"]:
    row = execution_pdf[execution_pdf["variant"] == variant_str].iloc[0]
    logged_dict_list.append(log_trial_dict(exp_config, "robustness", {"step": "step04", "variant": variant_str, "candidate_name": "E1M1", "span": "dev"},
                                           {col: row[col] for col in execution_pdf.columns if col != "variant"}, False, fold_row_in={"valid_start": date1, "valid_end": date2},
                                           note_str_in="exp13 step04 execution robustness (frozen E1M1)"))
# LOG THE SUMMARY ENTRY (THE VERDICT)
logged_dict_list.append(log_trial_dict(exp_config, "summary", {"step": "step04", "candidate_name": "E1M1", "verdict": verdict_dict["verdict_str"]},
                                       {**verdict_input_dict, **verdict_dict, "frozen_dev_metric_dict": frozen_metric_dict, "concentration_dict": concentration_dict,
                                        "half_log_excess_list": half_pdf["log_excess"].tolist(), "reason_split_dict": reason_pdf.set_index("reentry_trigger")["product_s_over_r"].to_dict()},
                                       False, fold_row_in={"valid_start": date1, "valid_end": date2}, note_str_in="exp13 step04 summary and verdict (PROTOCOL.md §8)"))
print(f"Trials logged:\t{len(logged_dict_list)}\tfirst logged_ts {logged_dict_list[0]['logged_ts']}\ttest_window_touched: {sorted({row['test_window_touched'] for row in logged_dict_list})}")

Trials logged:	3	first logged_ts 2026-10-09T20:29:30	test_window_touched: [False]


In [11]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
